In [2]:
import os
import sys
sys.path.insert(0, "/home/thomasb/")

import numpy as np
import time
import matplotlib.pyplot as plt
import numba as nb
from scipy.stats import median_abs_deviation
from scipy import linalg
import rfitools
import importlib
from scipy.ndimage import median_filter
import os
from datetime import datetime
from astropy.coordinates import EarthLocation, SkyCoord, AltAz
from astropy.time import Time
import matplotlib.colors as mcolors
from albatros_analysis.src.correlations import timing_solution_class as tsc

import map_utils as mutils

In [3]:
tsobj = tsc.TimingSolution(1753200150, '/scratch/thomasb/timing_solution')

In [4]:
path_satdata = '/scratch/mohanagr/map_output'
#path_satdata = '/scratch/thomasb/pipeline_test_phases/data_without_clock'

npasses = 13

In [5]:
#regular data
osamp = 64  
acclen = 512  
ntimes = 160573
chanstart, chanend = 360, 392

#sat passes
#osamp = 4    #sat passes
#acclen = 1024 #sat passes
#chanstart, chanend = 1834, 1852

start_specnum = 1273601 #both

dt = acclen*osamp * 4096/250e6
UTC_offset = tsobj.UTC_offset
UTC_per_spec = tsobj.UTC_per_spec
unix_start = start_specnum * UTC_per_spec + UTC_offset
fine_tarr = unix_start + (np.arange(ntimes) + 0.5) * dt

freqs = np.arange(chanstart*osamp,chanend*osamp)/osamp * 250e6/4096 # NOT aliased

print('Freqs shape', freqs.shape)
print('df', 250e6/(4096*osamp), freqs[1]-freqs[0])
print('fine tarr shape', fine_tarr.shape)
print('dt', dt, fine_tarr[1]-fine_tarr[0])

Freqs shape (2048,)
df 953.67431640625 953.67431640625
fine tarr shape (160573,)
dt 0.536870912 0.5368709564208984


In [7]:
for pidx in range(npasses):

    #open npz
    d = np.load(os.path.join(path_satdata, f'jul22_satpass_{pidx}.npz')) #old dumped ones
    
    #d = np.load(os.path.join(path_satdata, f'satpass_{pidx}_noclock.npz'))

    #science band data
    vis_pass = d['data']
    print('vis_pass shape', vis_pass.shape)
    mask_pass = d['mask']
    print('mask_pass shape', mask_pass.shape)
    unix_pass = d['utc']
    print('utc pass shape', unix_pass.shape)
    print('start of pass', unix_pass[0])
    
    #satellite phase test data
    # vis_pass = d['vis']
    # vis_pass = vis_pass.transpose(0,2,1)
    # print('vis_pass shape', vis_pass.shape)
    # mask_pass = d['mask']
    # mask_pass = mask_pass.transpose(0,2,1)
    # print('mask_pass shape', mask_pass.shape)
    # unix_pass = d['times']
    # print('utc pass shape', unix_pass.shape)
    # print('start of pass', unix_pass[0])
    # print('dt', unix_pass[1]-unix_pass[0])
    # freqs = d['freqs']
    # print('freqs shape', freqs.shape)
    # print('start freq', freqs[0])


    print('Unix times around satellite pass', unix_pass.shape)
    print('Duration of satellite pulse (s)', unix_pass[-1]-unix_pass[0])

    # get the clock delays for each baseline at the visibility times
    clock_delays_pass = tsobj.interpolate_delay2(unix_pass, extrapolate=True)
    clock_delays_pass, b = tsobj.all_blines(clock_delays_pass)
    clock_delays_pass *= 1e-9 #turn into ns
    print('clock delays shape', clock_delays_pass.shape)
    print(clock_delays_pass[:,0])

    sys.exit()
    #correct for clock
    # vis_pass *= np.exp(
    #     2j*np.pi
    #     *freqs[None, :, None] # (1, nchans, 1)
    #     *clock_delays_pass.T[:, None, :] # (ntimes, 1, nbl)
    #     )

    vis_pass *= np.exp(
        2j*np.pi
        *freqs[None, None, :] # (BD, nchans, BD)
        *clock_delays_pass.T[:, :, None] # (ntimes, BD, nbl)
        )

    print('pass visibility shape', vis_pass.shape)
    print('pass mask shape', mask_pass.shape)
    print('pass times shape', unix_pass.shape)
    print('pass frequencies shape', freqs.shape)

    path_data = '/scratch/thomasb/mapmaking_dumps/all_sats_science_band_flipped'
    #path_data = '/scratch/thomasb/mapmaking_dumps/clock_corrector_testing'

    os.makedirs(path_data, exist_ok=path_data)
    np.savez(
        os.path.join(path_data, f'satpass_{pidx}.npz'),
        vis = vis_pass,
        mask = mask_pass,
        times = unix_pass,
        freqs = freqs,
    )

vis_pass shape (246, 21, 2048)
mask_pass shape (246, 21, 2048)
utc pass shape (246,)
start of pass 1753204686.645253
Unix times around satellite pass (246,)
Duration of satellite pulse (s) 131.53337359428406
number of baselines (containing ref ant) in tau data: 6
number of desired interpolation unix times: 246
number of total data unix times (1560,)
clock delays shape (21, 246)
[-3.05555434e-06 -2.14171603e-05  7.76885816e-06 -3.63449347e-06
 -4.09260468e-06  8.42041186e-06 -1.83616059e-05  1.08244125e-05
 -5.78939131e-07 -1.03705033e-06  1.14759662e-05  2.91860184e-05
  1.77826668e-05  1.73245556e-05  2.98375721e-05 -1.14033516e-05
 -1.18614628e-05  6.51553700e-07 -4.58111200e-07  1.20549053e-05
  1.25130165e-05]


SystemExit: 